In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import json
import time
import os

from src.neuron_heads import head_attribution_over_all_data
from src.datahandlers import ActivatingDataset
from src.utils import tuple_str_to_tuple
from src.neuron_explain import generate_classify_prompt_dict

In [3]:
head_attributions_folder = "4_new_head_attributions"
suffix = ""
if head_attributions_folder == "4_head_attributions":
    suffix += "_baseline"

judge_model = "gpt-5.1"

In [4]:


# filename = "2023-10-13_13-51-37_gpt2-small_test"
# filename = "2023-10-13_14-02-06_gpt2-medium_test"
filename = "2026-06-11_23-15-48_gpt2-small_test"

with open(f'../experiment_data/{head_attributions_folder}/{filename}.json') as f:
    head_attributions = json.load(f)

trimmed_texts_filename = head_attributions['prior_filename']
with open(f'../experiment_data/3_trimmed_texts/{trimmed_texts_filename}.json') as f:
    trimmed_texts = json.load(f)

max_activating_filename = trimmed_texts['prior_filename']
with open(f'../experiment_data/2_max_activating_texts/{max_activating_filename}.json') as f:
    max_activating = json.load(f)

neuron_filename = max_activating['prior_filename']
with open(f'../experiment_data/1_next_token_neurons/{neuron_filename}.json') as f:
    neurons_data = json.load(f)

neurons = [tuple_str_to_tuple(neuron_str) for neuron_str in head_attributions['head_attributions'].keys()]

neuron_to_token = {tuple_str_to_tuple(neuron_str): token_data['token'] for neuron_str, token_data in neurons_data['neurons'].items()}

# Parameters
parameters = head_attributions['parameters']
model_name = parameters['model_name']

In [5]:
# Load explanations

import json

folder_path = "../experiment_data/6_head_explanations"
filename_jsonl = "2026-06-25_17-53-49_gpt2-small_train_gpt-5.1_results"
# Load the jsonl
with open(os.path.join(folder_path, f'{filename_jsonl}.jsonl'), 'r') as f:
    lines = f.readlines()
    lines = [json.loads(line) for line in lines]

x = lines[0]
# print(x[0]['messages'])
exp_prompt_to_exp = {x[0]['messages'][0]['content']: x[1]['choices'][0]['message']['content'] for x in lines}

filename_json = "2026-06-25_17-53-49_gpt2-small_train_gpt-5.1_prompts_dict"
with open(os.path.join(folder_path, f'{filename_json}.json')) as f:
    nh_to_exp_prompt = json.load(f)

nh_to_exp_prompt = {tuple_str_to_tuple(k): v for k, v in nh_to_exp_prompt.items()}

nh_to_exp = {nh: exp_prompt_to_exp[exp_prompt] for nh, exp_prompt in nh_to_exp_prompt.items()}

In [7]:
neuron_to_prompts = {tuple_str_to_tuple(neuron): 
                     list(head_attribution.keys()) 
                     for neuron, head_attribution in head_attributions['head_attributions'].items()}

In [8]:
neuron_to_prompts

{(9, 5): ["'m sure others do as",
  ' not just your palms that get calluses but your mind as',
  ' not only does the Council listen, but it would appear the Government is listening as',
  ' certificate will fix that problem as it applies to not only the main domain but to the subdomains under it as',
  ' leads to increasing wealth inequality for lots of other reasons',
  ' messed up, but I was resolved to break that whole cycle of abuse.  JANUSZEWSKI: Do you think you did?  MILLER: Yeah, I did.  JANUSZEWSKI: Yeah, I think you did,',
  'S NAME.  I know my broker cares about me because he always signs his first name and has his staff sign their first names as',
  ' having a good time, and the kids are having a good time',
  ' I’m so grateful to have studied at such amazing schools, and I want everyone else who is interested in working in wellness to have an amazing time studying as',
  ' clients love us, and we think you will',
  ' But just as Pyongyang seems impervious to all that Washi

In [9]:
head_attributions['head_attributions']

{'(9, 5)': {"'m sure others do as": [118, 101, 100, 92, 87, 69, 72, 65],
  ' not just your palms that get calluses but your mind as': [88,
   118,
   34,
   93,
   45,
   80],
  ' not only does the Council listen, but it would appear the Government is listening as': [118,
   34,
   88,
   69,
   83,
   47,
   106,
   93,
   9,
   44,
   33,
   16,
   50,
   24,
   92,
   32,
   101,
   22,
   80],
  ' certificate will fix that problem as it applies to not only the main domain but to the subdomains under it as': [118,
   88,
   93,
   32,
   68,
   33,
   83,
   24,
   69,
   16,
   8,
   97,
   106,
   61],
  ' leads to increasing wealth inequality for lots of other reasons': [118,
   69,
   100,
   88,
   34,
   23,
   32],
  ' messed up, but I was resolved to break that whole cycle of abuse.  JANUSZEWSKI: Do you think you did?  MILLER: Yeah, I did.  JANUSZEWSKI: Yeah, I think you did,': [118,
   78,
   101,
   88,
   34,
   83,
   32,
   87,
   69],
  'S NAME.  I know my broker cares

In [10]:
sum = 0
for neuron in neuron_to_prompts.keys():
    sum += len(neuron_to_prompts[neuron])
sum

1093

In [11]:
gpt_4_prompts_dict = generate_classify_prompt_dict(
    nh_to_exp = nh_to_exp,
    neuron_to_prompts=neuron_to_prompts,
    neuron_to_token=neuron_to_token,
    max_per_nh=10,
)

In [12]:
len(gpt_4_prompts_dict)

325

In [13]:
gpt_4_prompts = []
for nh, exp in gpt_4_prompts_dict.items():
    gpt_4_prompts += list(exp.keys())

len(gpt_4_prompts)

3176

In [14]:
gpt_4_prompts

['The head activates on patterns of *contrastive or additive parallelism* where a second item is being added to a first one, especially with “not only/just … but (for) X as too” or similar “not X but Y as too” structures. In other words, it fires when “too” (or “as too”) is used to extend a property/benefit from one entity to another in a parallel clause, rather than when “too/so too” is used in more generic additive or comparative ways.\nIs the given example an active example? (Yes/No)\n\nExample: """\n\'m sure others do as too\n"""\nAnswer: ',
 'The head activates on patterns of *contrastive or additive parallelism* where a second item is being added to a first one, especially with “not only/just … but (for) X as too” or similar “not X but Y as too” structures. In other words, it fires when “too” (or “as too”) is used to extend a property/benefit from one entity to another in a parallel clause, rather than when “too/so too” is used in more generic additive or comparative ways.\nIs th

In [15]:
jobs = [
            {"model":judge_model,
            "messages":[{"role": "user", "content": gpt_4_prompt}],
            "reasoning_effort": "none",
            "max_completion_tokens":20 # Just for safety
        } for gpt_4_prompt in gpt_4_prompts]

folder_path = "../experiment_data/7_explanation_eval"
os.makedirs(folder_path, exist_ok=True)

filepath = os.path.join(folder_path, f'{filename + suffix}_{judge_model}.jsonl')
if os.path.isfile(filepath):
    raise Exception("File already exists!")

with open(filepath, "w") as f:
    for job in jobs:
        json_string = json.dumps(job)
        f.write(json_string + "\n")

In [16]:
output = {
    'parameters': parameters,
    'prior_filename': filename,
    'classify_filepath': filepath,
    'classify_prompts': {str(k):v for k,v in gpt_4_prompts_dict.items()}
}

# Save json to ../experiment_data/2_max_activating_texts
timestamp = time.strftime("%Y-%m-%d_%H-%M-%S", time.localtime(int(time.time())))
folder_path = "../experiment_data/7_explanation_eval"
new_filename = f"{timestamp}_{model_name + suffix}_test_{judge_model}.json"
filepath = os.path.join(folder_path, new_filename)
if os.path.isfile(filepath):
    raise Exception("File already exists!")

with open(filepath, 'w') as f:
    json.dump(output, f)

In [17]:
prompt_token_cost = 0.05/1000000

import tiktoken
encoding = tiktoken.get_encoding("o200k_base")

# Count the number of tokens in `gpt_4_prompts`
prompt_tokens = 0
for gpt_4_prompt in gpt_4_prompts:
    prompt_tokens += len(encoding.encode(gpt_4_prompt))



print(prompt_tokens)
print(prompt_tokens * prompt_token_cost)

477482
0.023874100000000002
